# MetroLab · 01 — Del feed de Waze al panel de 5 minutos

Reemplaza los pasos 1 a 8 de `05_PREPARAR_CARGAR_DF` y `01_EXPLORACION_WAZE`. La red del corredor (tronco, abscisas, tramo activo) **no se recalcula**: se lee de `segmentos_corredor_v2` y `puntos_corredor_v2`, que ya están en Azure.

| Paso | Qué hace | Sale a `03_MODELACION/v3/` |
|---|---|---|
| 1 | Lee **solo los archivos nuevos** del feed (`jams_autosur_json`) y separa capturas y vértices del corredor | `capturas`, `jams_corredor` |
| 2 | Ubica cada vértice sobre el corredor (abscisa) con una tabla de coordenadas ya resueltas | `ubicacion_vertices` |
| 3 | Arma una fila por foto de jam y marca cierres (persistente / puntual) | `cierres_uuid_dia` |
| 4 | Intervalo de cada foto de trancón (reglas del paso 5) | — |
| 5 | Corta cada trancón en fragmentos por segmento y calcula su estado en cada captura | `estado_segmento_captura`, `vel_libre_segmento` |
| 6 | **Panel de 5 minutos**: segmento × calzada × ventana | `panel_5min` |
| 7 | Control: qué archivos y fechas quedaron procesados | `control/` |

**Modos** (celda 0.1):
- `"completo"`: la **primera vez** (y si se cambia alguna regla). Procesa todo el feed.
- `"actualizar"`: cada vez que llegan datos nuevos. Procesa solo las fechas nuevas o que cambiaron.

**Corrección importante frente a la versión anterior.** Antes, la lista de capturas quedó en hora UTC y los jams en hora de Bogotá, y en el paso 8.5 cada jam buscaba su captura por el **segundo exacto**: desde que Waze entrega segundos variables, la mayoría de jams no encontraba su ventana y se perdía. Ahora todo usa un solo reloj (`ts_s` = segundos en hora de Bogotá) y cada captura cae en su ventana por cálculo directo.

Todas las tablas nuevas quedan en `03_MODELACION/v3/`, así que las anteriores no se borran. Correr las celdas **en orden**.

## 0. Parámetros, Spark y Azure

In [ ]:
# ============================================================
# 0.1 Parámetros de la corrida
# ============================================================
MODO = "actualizar"     # "completo": la primera vez o para reconstruir todo
                        # "actualizar": solo fechas nuevas o que cambiaron en el feed
FORZAR_DESDE = None     # opcional, "AAAA-MM-DD": en modo actualizar, vuelve a procesar desde esa fecha
RECALCULAR_ATRIBUTOS = False   # True: vuelve a calcular la ciudad (city) y la vía (NomVia2) de cada segmento
VEL_LIBRE_FIJA = 40.0   # km/h que se asignan donde no hay jam (flujo libre), igual en todo el corredor.
                        # None = usar la mediana histórica de cada segmento (método anterior).
                        # Si se cambia este valor, hay que correr en MODO = "completo".
VALIDAR = MODO == "completo"   # los conteos de validación cuestan tiempo: solo en modo completo

assert MODO in ("completo", "actualizar"), "MODO debe ser 'completo' o 'actualizar'"
print("Modo:", MODO)

In [ ]:
# ============================================================
# 0.2 Spark y Azure (misma configuración de 00_Conexion_Azure_PySpark)
# ============================================================
import os, sys, subprocess, time, json, shutil, gc, glob
import importlib.metadata
try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    import pyspark
import psutil

# ---- 1. Entorno: Colab (con Google Drive) o GitHub Actions ----
try:
    from google.colab import userdata
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def secreto(*nombres):
    """Primer secreto que exista: Secretos de Colab (llave) o variables de entorno (GitHub Actions)."""
    for n in nombres:
        try:
            v = userdata.get(n) if EN_COLAB else os.environ.get(n)
        except Exception:
            v = None
        if v:
            return v
    raise ValueError(f"Falta el secreto {nombres[0]} (revisa la llave de Secretos de Colab)")

CONTENT = "/content" if EN_COLAB else os.path.abspath("trabajo")
os.makedirs(CONTENT, exist_ok=True)

PROJECT_DIR = None                               # carpeta del proyecto en Google Drive
if EN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Waze_GD"
    if os.path.isdir(PROJECT_DIR):
        PATH_NOTEBOOKS, PATH_SALIDAS, PATH_CONFIG = (f"{PROJECT_DIR}/{c}" for c in ("Notebooks", "Salidas", "Config"))
        os.makedirs(PATH_SALIDAS, exist_ok=True)
        print("✅ Google Drive conectado · proyecto:", PROJECT_DIR)
    else:
        print("⚠️ No encontré", PROJECT_DIR, "(se sigue sin Drive)")
        PROJECT_DIR = None

# ---- 2. Recursos (Colab Pro: 8 CPU y ~51 GB → 8 núcleos y ~30 GB para Spark) ----
CPU_SPARK = os.cpu_count()
RAM_GB = psutil.virtual_memory().total / 1024**3
MEMORIA_SPARK = f"{max(4, int(RAM_GB * 0.6))}g"
PARTICIONES_SHUFFLE = CPU_SPARK * 8               # 8 CPU → 64, como en el script de conexión
TMP_LOCAL = f"{CONTENT}/tmp_metrolab"             # temporales en el disco local (mucho más rápido que Azure)
os.makedirs(TMP_LOCAL, exist_ok=True)

# ---- 3. Conector Hadoop-Azure ----
HADOOP_VERSION = "3.4.1"
AZURE_PACKAGE = f"org.apache.hadoop:hadoop-azure:{HADOOP_VERSION}"
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {AZURE_PACKAGE} pyspark-shell"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"        # evita conflictos de red de Spark en Colab
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ---- 4. Spark: reutiliza la sesión si ya existe ----
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F, types as T
import numpy as np
import pandas as pd

spark = SparkSession.getActiveSession()
if spark is not None:
    print("✅ Se reutiliza la sesión Spark existente")
else:
    spark = (
        SparkSession.builder
        .appName("MetroLab_01_Panel")
        .master(f"local[{CPU_SPARK}]")
        .config("spark.driver.memory", MEMORIA_SPARK)
        .config("spark.driver.maxResultSize", "2g")
        .config("spark.sql.shuffle.partitions", str(PARTICIONES_SHUFFLE))
        .config("spark.default.parallelism", str(CPU_SPARK * 2))
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.local.dir", f"{TMP_LOCAL}/spark")
        .getOrCreate()
    )
    print("✅ Nueva sesión Spark iniciada")
spark.sparkContext.setLogLevel("WARN")

# Ajustes de la línea MetroLab (se aplican también si la sesión ya existía)
for k, v in {
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    # Un solo reloj: 'ts_s' y 'v' son HORA DE BOGOTÁ guardada como si fuera UTC;
    # con la sesión en UTC, Spark muestra esas horas tal cual, sin correrlas 5 horas
    "spark.sql.session.timeZone": "UTC",
    "spark.sql.sources.partitionOverwriteMode": "dynamic",              # reescribe solo las fechas tocadas
    "spark.sql.sources.partitionColumnTypeInference.enabled": "false",  # 'fecha' se lee como texto
    "spark.sql.execution.arrow.pyspark.enabled": "true",
}.items():
    spark.conf.set(k, v)

# ---- 5. Autenticación Azure ADLS Gen2 ----
account    = secreto("AZURE_STORAGE_ACCOUNT", "storage_account_name")
key        = secreto("AZURE_STORAGE_KEY", "storage_account_key")
filesystem = secreto("AZURE_FILE_SYSTEM", "file_system_name")
endpoint   = f"{account}.dfs.core.windows.net"
hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set(f"fs.azure.account.auth.type.{endpoint}", "SharedKey")
hconf.set(f"fs.azure.account.key.{endpoint}", key)
del key

BASE      = f"abfss://{filesystem}@{endpoint}/Waze_feed_tvt_parquet"
RUTA_JAMS = f"{BASE}/salidas_intermedias/02_ASIGNACION_ESPACIAL/jams_autosur_json"
SALIDA    = f"{BASE}/salidas_intermedias/03_MODELACION"
V3        = f"{SALIDA}/v3"                   # todo lo nuevo queda aquí; lo anterior no se toca

OFFSET_S  = 5 * 3600     # Bogotá = UTC − 5 (Colombia no tiene horario de verano)
VENTANA_S = 300          # ventanas de 5 minutos

# ---- Utilidades de archivos ----
Path = spark._jvm.org.apache.hadoop.fs.Path
def fs_de(ruta):  return Path(ruta).getFileSystem(hconf)
def existe(ruta): return fs_de(ruta).exists(Path(ruta))
def borrar(ruta):
    if existe(ruta):
        fs_de(ruta).delete(Path(ruta), True)
def local(nombre): return f"file://{TMP_LOCAL}/{nombre}"
def guardar_local(df, nombre):
    """Escribe en el disco de Colab y vuelve a leer: corta el plan de Spark y evita recalcular."""
    df.write.mode("overwrite").parquet(local(nombre))
    return spark.read.parquet(local(nombre))
def fechas_de(ruta):
    """Fechas guardadas en una tabla partida por 'fecha' (solo lista carpetas, no lee datos)."""
    if not existe(ruta):
        return []
    return sorted(s.getPath().getName().split("=", 1)[1] for s in fs_de(ruta).listStatus(Path(ruta))
                  if s.isDirectory() and s.getPath().getName().startswith("fecha="))
def fecha_de(col_segundos):
    return F.date_format(F.timestamp_seconds(col_segundos), "yyyy-MM-dd")

# ---- 6. Validación de conexión y rutas ----
print("\n=== CONFIGURACIÓN SPARK ===")
print("Versión:", spark.version, "· Master:", spark.sparkContext.master,
      "· Memoria:", spark.sparkContext.getConf().get("spark.driver.memory", MEMORIA_SPARK),
      "· Particiones:", spark.conf.get("spark.sql.shuffle.partitions"))
print("\n=== VALIDACIÓN DE CONEXIÓN AZURE ===")
rutas_ok = {}
for nombre, ruta in [("Waze_feed_tvt_parquet", BASE), ("02_ASIGNACION_ESPACIAL/jams_autosur_json", RUTA_JAMS),
                     ("03_MODELACION", SALIDA)]:
    rutas_ok[nombre] = existe(ruta)
    print(("✅ OK" if rutas_ok[nombre] else "❌ NO EXISTE"), "|", nombre)
if not all(rutas_ok.values()):
    raise FileNotFoundError("Revisa las rutas que no existen (o las credenciales de Azure)")
print("✅ CONEXIÓN AZURE VALIDADA · salidas nuevas en:", V3)

In [ ]:
# ============================================================
# 0.3 Insumos fijos del corredor (ya calculados en Azure)
# ============================================================
from scipy.spatial import cKDTree

seg_pd = (spark.read.parquet(f"{SALIDA}/segmentos_corredor_v2")
               .select("sentido", "FROM_TO", "orden_seg", "longitud_m", "abs_ini_m", "abs_fin_m", "activo")
               .toPandas())
seg_sp = spark.createDataFrame(seg_pd)

pts_pd = (spark.read.parquet(f"{SALIDA}/puntos_corredor_v2")
               .select("sentido", "FROM_TO", "x", "y", "abscisa_m", "pos_seg_m", "espina_m")
               .dropDuplicates(["sentido", "FROM_TO", "x", "y"])
               .toPandas())

FIN_CALZADA = seg_pd.groupby("sentido").abs_fin_m.max().to_dict()
LAT0 = 4.6                                    # latitud de referencia para pasar grados a metros

# Calles que cuentan como corredor (paso 5.5)
CALLES_CORREDOR = [
    "Autopista Sur > Occidente", "Autopista Sur > Oriente", "Autopista Sur >Occidente",
    "AP Bogotá-Granada/ RN40-05 >Occidente", "AP Granada-Bogotá/ RN40-05 >Oriente",
    "AP Bogotá-Granada / RN40-05", "Paralela Autopista Sur", "Entrada a Bogotá",
]
PREFIJOS_CORREDOR = ["Peaje Chusacá"]          # el nombre incluye la tarifa, que cambia
UMBRAL_PERSISTENTE_H = 24                      # cierres de 24 h o más = persistentes
MIN_TRANCONES = 30                             # mínimo de trancones para la velocidad libre del segmento
MAX_HUECO_S = 600                              # ventanas vacías: se completan con capturas a ≤ 10 min

print("Segmentos del tronco:", len(seg_pd), "· activos:", int(seg_pd.activo.sum()))
print("Puntos de 3 m:", f"{len(pts_pd):,}")
print("Largo por calzada (km):", {k: round(v / 1000, 2) for k, v in FIN_CALZADA.items()})

## 1. Carga de `jams_autosur_json` (la tabla que actualiza Fabric)

`jams_autosur_json` (en `02_ASIGNACION_ESPACIAL`) es la entrada de toda la línea: Fabric la actualiza a diario con los jams ya asignados a los segmentos. Antes, el cuaderno 05 leía `df_mdt`, una **copia fija** que se sacó una sola vez de esa tabla (por eso todo se quedaba en el 23 de octubre). Ahora se lee directo de `jams_autosur_json` en cada corrida.

Se listan sus archivos y se comparan con los ya procesados (ruta y fecha de modificación). Solo se leen los nuevos o cambiados, con el **esquema fijo** (no se adivinan tipos: leer JSON así es varias veces más rápido).

De cada fila se calcula `ts_s` = segundos de `endTimeMillis` **menos 5 horas** (hora de Bogotá) y su `fecha`.
- `capturas`: todas las horas de captura del feed (todas las calles): dicen si Waze estaba tomando datos.
- `jams_corredor`: solo los vértices de los 600 segmentos del tronco y en su mismo sentido (antes pasos 1.x y 3.1).

Si un archivo trae una fecha que ya estaba guardada, se une con lo anterior de esa fecha y se quitan repetidos.

In [ ]:
# ============================================================
# 1.1 ¿Qué archivos hay que leer?
# ============================================================
RUTA_CTRL_ARCH = f"{V3}/control/archivos"

def listar_archivos(ruta):
    it = fs_de(ruta).listFiles(Path(ruta), True)
    filas = []
    while it.hasNext():
        st = it.next()
        ruta_rel = st.getPath().toString().split(ruta.rstrip("/").split("/")[-1], 1)[-1]
        # fuera: archivos de control (_SUCCESS, _delta_log/…, .crc) y vacíos
        if any(p.startswith(("_", ".")) for p in ruta_rel.split("/") if p) or st.getLen() == 0:
            continue
        filas.append((st.getPath().toString(), int(st.getModificationTime()), int(st.getLen())))
    return pd.DataFrame(filas, columns=["ruta", "mod_ms", "bytes"])

if MODO == "actualizar" and not existe(RUTA_CTRL_ARCH):
    raise SystemExit("Aún no existe la línea v3: corre primero este cuaderno en Colab con MODO = 'completo'.")
archivos = listar_archivos(RUTA_JAMS)
if MODO == "actualizar" and existe(RUTA_CTRL_ARCH):
    vistos = spark.read.parquet(RUTA_CTRL_ARCH).toPandas()
    m = archivos.merge(vistos[["ruta", "mod_ms"]], on=["ruta", "mod_ms"], how="left", indicator=True)
    nuevos = archivos[(m["_merge"] == "left_only").to_numpy()]
else:
    nuevos = archivos

if archivos.empty:
    raise SystemExit(f"No encontré archivos en {RUTA_JAMS}: revisa la ruta")
ultimo_mod = pd.to_datetime(archivos.mod_ms.max(), unit="ms") - pd.Timedelta(hours=5)
print(f"jams_autosur_json: {len(archivos):,} archivos ({archivos.bytes.sum() / 1024**3:.2f} GB)")
print(f"  Última actualización de Fabric: {ultimo_mod:%Y-%m-%d %H:%M} (hora Bogotá)")
print(f"  Archivos nuevos o cambiados por leer: {len(nuevos):,} ({nuevos.bytes.sum() / 1024**3:.2f} GB)")

In [ ]:
# ============================================================
# 1.1b Fechas de jams_autosur_json frente a lo ya procesado en v3
# ============================================================
def rango_archivo(ruta):
    """Primer y último dato (hora Bogotá) dentro de un archivo de jams_autosur_json."""
    r = (spark.read.schema(ESQUEMA_MIN).json(ruta)
              .agg(F.min("endTimeMillis").alias("a"), F.max("endTimeMillis").alias("b")).first())
    a = pd.to_datetime(r.a, unit="ms") - pd.Timedelta(hours=5) if r.a else None
    b = pd.to_datetime(r.b, unit="ms") - pd.Timedelta(hours=5) if r.b else None
    return a, b

ESQUEMA_MIN = T.StructType([T.StructField("endTimeMillis", T.LongType())])
por_nombre = archivos.sort_values("ruta")
filas = []
for etiqueta, ruta in [("primer archivo", por_nombre.ruta.iloc[0]), ("último archivo", por_nombre.ruta.iloc[-1])]:
    a, b = rango_archivo(ruta)
    filas.append({"archivo": etiqueta, "nombre": ruta.split("/")[-1][:60],
                  "primer dato": f"{a:%Y-%m-%d %H:%M}" if a is not None else "-",
                  "último dato": f"{b:%Y-%m-%d %H:%M}" if b is not None else "-"})
print("jams_autosur_json (Fabric):")
display(pd.DataFrame(filas))

RUTA_CAP = f"{V3}/capturas"
hechas = fechas_de(RUTA_CAP)
if hechas:
    ts_ult = spark.read.parquet(RUTA_CAP).filter(F.col("fecha") == hechas[-1]).agg(F.max("ts_s")).first()[0]
    print(f"Ya procesado en v3: {hechas[0]} a {pd.to_datetime(ts_ult, unit='s'):%Y-%m-%d %H:%M} "
          f"({len(hechas)} fechas)")
else:
    print("Ya procesado en v3: nada todavía (corre en MODO = 'completo')")

In [ ]:
# ============================================================
# 1.2 Leer lo nuevo, separar capturas y vértices del tronco
# ============================================================
ESQUEMA = T.StructType([
    T.StructField("FROM_TO", T.StringType()),   T.StructField("sentido", T.StringType()),
    T.StructField("uuid", T.LongType()),        T.StructField("endTimeMillis", T.LongType()),
    T.StructField("line_pos", T.LongType()),    T.StructField("line_x", T.DoubleType()),
    T.StructField("line_y", T.DoubleType()),    T.StructField("street", T.StringType()),
    T.StructField("speedKMH", T.DoubleType()),  T.StructField("delay", T.LongType()),
    T.StructField("length", T.LongType()),      T.StructField("level", T.LongType()),
    T.StructField("dist_m", T.DoubleType()),     T.StructField("city", T.StringType()),
])
RUTA_CAP = f"{V3}/capturas"
RUTA_JC  = f"{V3}/jams_corredor"

if MODO == "completo":
    for r in [RUTA_CAP, RUTA_JC]:
        borrar(r)

FECHAS_NUEVAS = []
if len(nuevos):
    t0 = time.time()
    tronco = seg_sp.select("sentido", "FROM_TO").withColumn("en_tronco", F.lit(True))
    crudo = (spark.read.schema(ESQUEMA).json(list(nuevos.ruta))
                  .filter(F.col("endTimeMillis").isNotNull())
                  .withColumn("ts_s", F.floor(F.col("endTimeMillis") / 1000).cast("long") - OFFSET_S)
                  .withColumn("fecha", fecha_de("ts_s")))
    # En modo actualizar solo interesa desde el día anterior al último guardado (o FORZAR_DESDE):
    # si Fabric reescribe la tabla completa, no se reprocesa toda la historia
    ya = fechas_de(RUTA_CAP) if MODO == "actualizar" else []
    if ya:
        desde = str((pd.Timestamp(ya[-1]) - pd.Timedelta(days=1)).date())
        if FORZAR_DESDE:
            desde = min(desde, FORZAR_DESDE)
        crudo = crudo.filter(F.col("fecha") >= desde)
        print(f"Último día ya procesado: {ya[-1]} → se toman datos desde {desde}")
    crudo = (crudo.join(F.broadcast(tronco), ["sentido", "FROM_TO"], "left")
                  .withColumn("en_tronco", F.coalesce(F.col("en_tronco"), F.lit(False)))
                  .drop("endTimeMillis"))
    leido = guardar_local(crudo, "feed_nuevo")          # el JSON se lee una sola vez
    ult_ts = leido.agg(F.max("ts_s")).first()[0]
    if ult_ts is not None:
        print(f"Último dato en jams_autosur_json: {pd.to_datetime(ult_ts, unit='s'):%Y-%m-%d %H:%M:%S} (hora Bogotá)")

    # Una fecha se procesa si es nueva o si trae capturas que no estaban
    # (si el proceso anterior reescribe todo el feed, las fechas viejas sin cambios se saltan)
    res_nuevo = leido.groupBy("fecha").agg(F.countDistinct("ts_s").alias("n_cap")).toPandas()
    if MODO == "actualizar" and existe(RUTA_CAP):
        res_viejo = (spark.read.parquet(RUTA_CAP).filter(F.col("fecha").isin(list(res_nuevo.fecha)))
                          .groupBy("fecha").agg(F.count("*").alias("n_cap_v")).toPandas())
        m = res_nuevo.merge(res_viejo, on="fecha", how="left")
        res_nuevo = m[m.n_cap_v.isna() | (m.n_cap > m.n_cap_v)]
    FECHAS_NUEVAS = sorted(res_nuevo.fecha)
    print(f"Leído en {time.time() - t0:.0f} s · fechas nuevas o con capturas nuevas: {len(FECHAS_NUEVAS)}"
          + (f" ({FECHAS_NUEVAS[0]} a {FECHAS_NUEVAS[-1]})" if FECHAS_NUEVAS else ""))
    leido = leido.filter(F.col("fecha").isin(FECHAS_NUEVAS))

    def fusionar(nuevo, ruta, llaves, nombre):
        """Une lo nuevo con lo guardado de las mismas fechas, quita repetidos y reescribe solo esas fechas."""
        if MODO == "actualizar" and existe(ruta):
            viejo = spark.read.parquet(ruta).filter(F.col("fecha").isin(FECHAS_NUEVAS))
            tipos = dict(nuevo.dtypes)
            viejo = viejo.select(*[F.col(c) if c in viejo.columns else F.lit(None).cast(tipos[c]).alias(c)
                                   for c in nuevo.columns])
            nuevo = nuevo.unionByName(viejo)
        df = guardar_local(nuevo.dropDuplicates(llaves), nombre)
        df.repartition("fecha").write.mode("overwrite").partitionBy("fecha").parquet(ruta)

    if FECHAS_NUEVAS:
        fusionar(leido.select("ts_s", "fecha").distinct(), RUTA_CAP, ["ts_s"], "capturas")
        fusionar(leido.filter("en_tronco").drop("en_tronco"), RUTA_JC,
                 ["sentido", "FROM_TO", "uuid", "ts_s", "line_pos"], "jams_corredor")
        print(f"Capturas y vértices del tronco guardados ({time.time() - t0:.0f} s)")

# Fechas que se procesan en esta corrida
if MODO == "completo":
    FECHAS = fechas_de(RUTA_CAP)
else:
    FECHAS = set(FECHAS_NUEVAS)
    if FORZAR_DESDE:
        FECHAS |= {f for f in fechas_de(RUTA_CAP) if f >= FORZAR_DESDE}
    FECHAS = sorted(FECHAS)

if not FECHAS:
    hechas = fechas_de(RUTA_CAP)
    if not hechas:
        raise ValueError("No hay datos en jams_autosur_json ni en v3: revisa la ruta RUTA_JAMS")
    FECHAS = [hechas[-1]]          # sin datos nuevos: se recalcula el último día para mantener la cadena al día
    print("No hay datos nuevos en jams_autosur_json. Se recalcula el último día procesado:", FECHAS[0])
print(f"\nFechas a procesar: {len(FECHAS)} ({FECHAS[0]} a {FECHAS[-1]})")

### 1.3 Ciudad y vía de cada segmento

Para filtrar la velocidad del corredor por **ciudad** y **vía** en el tablero, cada segmento recibe:
- `city`: la ciudad que Waze reporta con más frecuencia en los jams de ese segmento (Bogotá, Soacha, Sibaté, Granada…). Los segmentos sin jams toman la ciudad del segmento vecino.
- `NomVia2`: el nombre de la vía en la red (`NomVia2_First`, por ejemplo "Autopista Sur > Oriente").

Se calcula una vez (en modo completo o si la tabla no existe) y queda en `v3/atributos_segmento`.

In [ ]:
# ============================================================
# 1.3 Ciudad (city) y vía (NomVia2) de cada segmento
# ============================================================
RUTA_ATR = f"{V3}/atributos_segmento"
if MODO == "completo" or RECALCULAR_ATRIBUTOS or not existe(RUTA_ATR):
    t0 = time.time()
    esq_city = T.StructType([T.StructField("FROM_TO", T.StringType()), T.StructField("sentido", T.StringType()),
                             T.StructField("city", T.StringType())])
    moda = (spark.read.schema(esq_city).json(RUTA_JAMS)
                 .filter(F.col("city").isNotNull() & (F.trim("city") != ""))
                 .join(F.broadcast(seg_sp.select("sentido", "FROM_TO")), ["sentido", "FROM_TO"])
                 .groupBy("sentido", "FROM_TO", "city").count()
                 .withColumn("rk", F.row_number().over(Window.partitionBy("sentido", "FROM_TO").orderBy(F.desc("count"))))
                 .filter("rk = 1").select("sentido", "FROM_TO", "city", F.col("count").alias("vertices_city"))
                 .toPandas())
    via = (spark.read.parquet(f"{SALIDA}/red_corredor")
                .select("sentido", "FROM_TO", F.col("NomVia2_First").alias("NomVia2")).toPandas())
    atr = (seg_pd.merge(moda, on=["sentido", "FROM_TO"], how="left")
                 .merge(via, on=["sentido", "FROM_TO"], how="left")
                 .sort_values(["sentido", "orden_seg"]))
    # Segmentos sin jams: ciudad del vecino (anterior y, si no hay, siguiente) en la misma calzada
    atr["city_origen"] = np.where(atr.city.isna(), "vecino", "waze")
    atr["city"] = atr.groupby("sentido").city.transform(lambda s: s.ffill().bfill()).fillna("Sin dato")
    atr["NomVia2"] = atr.groupby("sentido").NomVia2.transform(lambda s: s.ffill().bfill()).fillna("Sin dato")
    atr["vertices_city"] = atr.vertices_city.fillna(0).astype("int64")
    spark.createDataFrame(atr).write.mode("overwrite").parquet(RUTA_ATR)
    print(f"Ciudad y vía por segmento calculadas ({time.time() - t0:.0f} s)")
atr_pd = spark.read.parquet(RUTA_ATR).toPandas()
NOMBRES = {"Occidente - Oriente": "Granada → Bogotá", "Oriente - Occidente": "Bogotá → Granada"}
print("Km por calzada, ciudad y vía (segmentos activos):")
display(atr_pd[atr_pd.activo]
        .assign(calzada=lambda d: d.sentido.map(NOMBRES))
        .groupby(["calzada", "city", "NomVia2"])
        .agg(segmentos=("FROM_TO", "size"), km=("longitud_m", lambda s: round(s.sum() / 1000, 1)),
             ciudad_de_vecino=("city_origen", lambda s: int((s == "vecino").sum())))
        .reset_index())

## 2. Ubicar cada vértice sobre el corredor

Igual que el paso 3.3: el punto de 3 m más cercano **dentro del mismo segmento y calzada**. Como los vértices de Waze se repiten (en todo el periodo hay alrededor de 1.100 coordenadas distintas), se guarda una tabla `ubicacion_vertices` y solo se calculan las coordenadas que no estaban. La abscisa ya sale corregida (paso 4.6), porque se usa `puntos_corredor_v2`.

In [ ]:
# ============================================================
# 2. Abscisa de cada vértice
# ============================================================
RUTA_UBIC = f"{V3}/ubicacion_vertices"
LLAVE_UBIC = ["sentido", "FROM_TO", "line_x", "line_y"]

jc = spark.read.parquet(RUTA_JC).filter(F.col("fecha").isin(FECHAS))
coords = jc.select(*LLAVE_UBIC).distinct().toPandas()

if MODO == "actualizar" and existe(RUTA_UBIC):
    ubic = spark.read.parquet(RUTA_UBIC).toPandas()
else:
    ubic = pd.DataFrame(columns=LLAVE_UBIC + ["abscisa_m", "pos_seg_m", "espina_m", "dist_pto_m"])

faltan = coords.merge(ubic[LLAVE_UBIC], on=LLAVE_UBIC, how="left", indicator=True)
faltan = faltan[faltan["_merge"] == "left_only"].drop(columns="_merge")

def a_metros(lon, lat):
    return np.column_stack([lon * np.cos(np.radians(LAT0)) * 111_320, lat * 110_574])

pts_g = {k: g for k, g in pts_pd.groupby(["sentido", "FROM_TO"])}
partes = []
for (sen, seg), g in faltan.groupby(["sentido", "FROM_TO"]):
    P = pts_g.get((sen, seg))
    if P is None:
        continue
    d, i = cKDTree(a_metros(P.x.values, P.y.values)).query(a_metros(g.line_x.values, g.line_y.values))
    cerca = P.iloc[i]
    partes.append(pd.DataFrame({
        "sentido": sen, "FROM_TO": seg, "line_x": g.line_x.values, "line_y": g.line_y.values,
        "abscisa_m": cerca.abscisa_m.values, "pos_seg_m": cerca.pos_seg_m.values,
        "espina_m": cerca.espina_m.values, "dist_pto_m": d}))

if partes:
    ubic = pd.concat([ubic] + partes, ignore_index=True)
    for c in ["line_x", "line_y", "abscisa_m", "pos_seg_m", "espina_m", "dist_pto_m"]:
        ubic[c] = ubic[c].astype("float64")
    spark.createDataFrame(ubic).write.mode("overwrite").parquet(RUTA_UBIC)

ubic_sp = spark.createDataFrame(ubic)
print(f"Coordenadas en esta corrida: {len(coords):,} · nuevas: {len(faltan):,} · tabla total: {len(ubic):,}")

vert = (jc.join(F.broadcast(ubic_sp), LLAVE_UBIC, "inner")
          .join(F.broadcast(seg_sp.select("sentido", "FROM_TO", "activo")), ["sentido", "FROM_TO"], "left"))
if VALIDAR:
    print("Vértices sin abscisa (esperado: 0):", jc.join(ubic_sp, LLAVE_UBIC, "left_anti").count())

## 3. Fotos de jam y cierres

Una fila por **foto** (un jam en una captura), como en 4.7. Además se guardan las coordenadas del primer y último vértice para decidir la calzada en 5.9c sin volver a leer los vértices.

**Cierres** (4.8): un jam en bloqueo (`level = 5` o retraso negativo) es *persistente* si el mismo `uuid` dura 24 h o más. La duración se calcula con todo lo guardado en `cierres_uuid_dia`, así un cierre que empezó días atrás se reconoce aunque solo se procesen las fechas nuevas.

In [ ]:
# ============================================================
# 3. Fotos de jam y tipo de cierre
# ============================================================
v = vert.dropDuplicates(["sentido", "uuid", "ts_s", "line_pos"])
w_foto = Window.partitionBy("sentido", "uuid", "ts_s").orderBy("line_pos")
v = (v.withColumn("salto_lp", F.col("line_pos") - F.lag("line_pos").over(w_foto))
      .withColumn("avance_m", F.col("abscisa_m") - F.lag("abscisa_m").over(w_foto)))

fotos = (v.groupBy("sentido", "uuid", "ts_s", "fecha")
          .agg(F.count("*").alias("n_vert"),
               F.min("line_pos").alias("lp_min"),
               F.max("line_pos").alias("lp_max"),
               F.min_by("abscisa_m", "line_pos").alias("cola_m"),
               F.max_by("abscisa_m", "line_pos").alias("cabeza_m"),
               F.min("abscisa_m").alias("abs_min"),
               F.max("abscisa_m").alias("abs_max"),
               F.sum(F.when(F.col("salto_lp") > 1, 1).otherwise(0)).alias("huecos_lp"),
               F.sum(F.when(F.col("avance_m") < -10, 1).otherwise(0)).alias("retrocesos"),
               F.avg(F.col("activo").cast("int")).alias("frac_activo"),
               F.min_by("speedKMH", "line_pos").alias("speedKMH"),
               F.round(F.min_by("delay", "line_pos") / 60, 2).alias("delay_minutes"),
               F.min_by("length", "line_pos").alias("length"),
               F.min_by("level", "line_pos").alias("level"),
               F.min_by("street", "line_pos").alias("street"))
          .withColumn("long_abs_m", F.col("abs_max") - F.col("abs_min"))
          .withColumn("bloqueo", (F.col("level").cast("int") == 5) | (F.col("delay_minutes") < 0)))
fotos = guardar_local(fotos, "fotos")

# ---- Duración de cada cierre con toda la historia ----
RUTA_CIERRES = f"{V3}/cierres_uuid_dia"
if MODO == "completo":
    borrar(RUTA_CIERRES)
(fotos.filter("bloqueo")
      .groupBy("uuid", "fecha")
      .agg(F.min("ts_s").alias("ts_min"), F.max("ts_s").alias("ts_max"), F.count("*").alias("capturas"))
      .repartition("fecha").write.mode("overwrite").partitionBy("fecha").parquet(RUTA_CIERRES))

dur = (spark.read.parquet(RUTA_CIERRES).groupBy("uuid")
            .agg(((F.max("ts_max") - F.min("ts_min")) / 3600).alias("duracion_h"))
            .withColumn("tipo_cierre", F.when(F.col("duracion_h") >= UMBRAL_PERSISTENTE_H, "persistente")
                                        .otherwise("puntual")))
fotos = (fotos.join(F.broadcast(dur.select("uuid", "tipo_cierre")), "uuid", "left")
              .withColumn("tipo_cierre", F.when(F.col("bloqueo"), F.col("tipo_cierre"))))
fotos = guardar_local(fotos, "fotos_cierre")

if VALIDAR:
    print("3a Fotos por calzada y tipo:")
    fotos.groupBy("sentido", "bloqueo", "tipo_cierre").count().orderBy("sentido", "bloqueo").show()

## 4. Intervalo de cada foto de trancón

Mismas reglas de los pasos 5.5 a 5.9:
- Se excluyen fotos de calles que no son del corredor, y las de **un solo vértice sin nombre de calle**.
- Varios vértices → intervalo de la primera a la última abscisa. Un vértice → se reconstruye con el largo de Waze (hacia adelante si es la cola, hacia atrás si no).
- Se excluyen las fotos cuyo nombre de calle contradice la calzada (por ejemplo, "> Oriente" en la calzada hacia Granada).
- Si una foto quedó en las dos calzadas, se queda en la que coincide con la **dirección del trancón** (coseno con la dirección de la vía).

In [ ]:
# ============================================================
# 4. Intervalo de cada foto (trancón, no cierre)
# ============================================================
calle_vacia = F.col("street").isNull() | (F.trim(F.col("street")) == "")
es_peaje = F.lit(False)
for p in PREFIJOS_CORREDOR:
    es_peaje = es_peaje | F.coalesce(F.col("street").startswith(p), F.lit(False))
fin_calzada = F.create_map(*[x for k, val in FIN_CALZADA.items() for x in (F.lit(k), F.lit(float(val)))])

largo = F.col("length").cast("double")
x = F.col("abs_min")
fc = (fotos.filter("NOT bloqueo")
           .withColumn("es_peaje", es_peaje)
           .withColumn("calle_corredor", F.col("street").isin(CALLES_CORREDOR) | calle_vacia | es_peaje)
           .withColumn("tipo_intervalo", F.when(~F.col("calle_corredor"), "excluida")
                                          .when(F.col("n_vert") > 1, "vertices")
                                          .when(calle_vacia, "excluida")          # 5.9a
                                          .otherwise("reconstruido"))
           .withColumn("ini_m", F.when(F.col("tipo_intervalo") == "vertices", F.col("abs_min"))
                                 .when(F.col("lp_min") == 0, x)
                                 .otherwise(F.greatest(x - largo, F.lit(0.0))))
           .withColumn("fin_m", F.when(F.col("tipo_intervalo") == "vertices", F.col("abs_max"))
                                 .when(F.col("lp_min") == 0, F.least(x + largo, fin_calzada[F.col("sentido")]))
                                 .otherwise(x))
           .withColumn("long_m", F.col("fin_m") - F.col("ini_m"))
           .withColumn("sentido_nombre", F.when(F.col("street").rlike(">\\s*Oriente"), "Occidente - Oriente")
                                          .when(F.col("street").rlike(">\\s*Occidente"), "Oriente - Occidente"))
           .withColumn("contradice_nombre", F.coalesce(F.col("sentido_nombre").isNotNull() &
                                                       (F.col("sentido_nombre") != F.col("sentido")), F.lit(False))))

validas = guardar_local(fc.filter("tipo_intervalo <> 'excluida' AND NOT contradice_nombre"), "validas")

# ---- 5.9c Fotos en las dos calzadas: decidir por la dirección del trancón ----
dup = (validas.groupBy("uuid", "ts_s").agg(F.countDistinct("sentido").alias("n"))
              .filter("n > 1").select("uuid", "ts_s"))
dir_jam = (vert.join(dup, ["uuid", "ts_s"], "left_semi")
               .groupBy("uuid", "ts_s")
               .agg(F.min_by("line_x", "line_pos").alias("x0"), F.min_by("line_y", "line_pos").alias("y0"),
                    F.max_by("line_x", "line_pos").alias("x1"), F.max_by("line_y", "line_pos").alias("y1")))
filas_dup = (validas.join(dup, ["uuid", "ts_s"]).join(dir_jam, ["uuid", "ts_s"])
                    .select("sentido", "uuid", "ts_s", "abs_min", "x0", "y0", "x1", "y1").toPandas())

fotos_corr = validas
if len(filas_dup):
    k = np.cos(np.radians(LAT0)) * 111_320
    ejes = {s: g.sort_values("abscisa_m")[["abscisa_m", "x", "y"]].to_numpy() for s, g in pts_pd.groupby("sentido")}
    def tangente(sentido, a):
        e = ejes[sentido]
        xs = np.interp([a - 30, a + 30], e[:, 0], e[:, 1])
        ys = np.interp([a - 30, a + 30], e[:, 0], e[:, 2])
        return (xs[1] - xs[0]) * k, (ys[1] - ys[0]) * 110_574
    t = np.array([tangente(s, a) for s, a in zip(filas_dup.sentido, filas_dup.abs_min)])
    jx, jy = (filas_dup.x1 - filas_dup.x0) * k, (filas_dup.y1 - filas_dup.y0) * 110_574
    filas_dup["coseno"] = (jx * t[:, 0] + jy * t[:, 1]) / (np.hypot(jx, jy) * np.hypot(t[:, 0], t[:, 1]))
    mejor = (filas_dup.dropna(subset=["coseno"]).sort_values("coseno", ascending=False)
                      .groupby(["uuid", "ts_s"]).head(1))
    mejor = mejor[mejor.coseno > 0]
    marcadas = filas_dup.merge(mejor[["sentido", "uuid", "ts_s"]].assign(se_queda=True),
                               on=["sentido", "uuid", "ts_s"], how="left")
    descartar = marcadas.loc[marcadas.se_queda.isna(), ["sentido", "uuid", "ts_s"]]
    if len(descartar):
        fotos_corr = validas.join(F.broadcast(spark.createDataFrame(descartar)),
                                  ["sentido", "uuid", "ts_s"], "left_anti")
    print(f"Fotos en las dos calzadas: {len(filas_dup) // 2:,} · filas descartadas: {len(descartar):,}")

fotos_corr = guardar_local(fotos_corr.withColumn("clave", F.concat_ws("|", F.col("uuid").cast("string"),
                                                                      F.col("ts_s").cast("string"))),
                           "fotos_corredor")
if VALIDAR:
    print("4a Fotos finales por calzada y regla:")
    fotos_corr.groupBy("sentido", "tipo_intervalo").count().orderBy("sentido", "tipo_intervalo").show()

## 5. Fragmentos y estado de cada segmento en cada captura

Pasos 6, 7, 8.2 y 8.3 en una sola pasada (sin guardar los 5 millones de fragmentos en Azure):
- Cada trancón se corta en un fragmento por segmento que toca (cajones de 1 km para no cruzar todo con todo).
- Cada fragmento hereda velocidad, nivel y una parte del retraso (proporcional a su largo).
- **Velocidad libre** (8.2): donde un segmento no tiene jam, va a `VEL_LIBRE_FIJA` (**40 km/h** en todo el corredor, celda 0.1). La mediana histórica de cada segmento se sigue calculando, solo como referencia y para marcar el peaje. Se calcula en modo completo y queda fija hasta la próxima reconstrucción.
- **Estado por segmento y captura** (8.3–8.4): largo y tiempo con trancón, velocidad del segmento y cierres.

In [ ]:
# ============================================================
# 5.1 Fragmentos y sus atributos (pasos 6 y 7)
# ============================================================
km_de = lambda c: F.floor(F.col(c) / 1000).cast("int")
seg_km = seg_sp.withColumn("km", F.explode(F.sequence(km_de("abs_ini_m"), km_de("abs_fin_m"))))

cols_foto = ["sentido", "clave", "ts_s", "fecha", "ini_m", "fin_m", "long_m", "tipo_intervalo", "es_peaje",
             "speedKMH", "delay_minutes", "length", "level"]
cruce = (fotos_corr.select(*cols_foto)
                   .withColumn("km", F.explode(F.sequence(km_de("ini_m"), km_de("fin_m"))))
                   .join(F.broadcast(seg_km), ["sentido", "km"])
                   .filter((F.col("abs_ini_m") <= F.col("fin_m")) & (F.col("abs_fin_m") >= F.col("ini_m")))
                   .dropDuplicates(["sentido", "clave", "FROM_TO"])
                   .drop("km")
                   .withColumn("long_frag_m", F.least("fin_m", "abs_fin_m") - F.greatest("ini_m", "abs_ini_m")))

frag_pos  = cruce.filter((F.col("long_m") > 0) & (F.col("long_frag_m") > 0))
w_cero    = Window.partitionBy("sentido", "clave").orderBy("orden_seg")
frag_cero = (cruce.filter(F.col("long_m") == 0)
                  .withColumn("rk", F.row_number().over(w_cero)).filter("rk = 1").drop("rk"))

w_jam  = Window.partitionBy("sentido", "clave")
vel_ms = F.col("vel_kmh") / 3.6
frag = (frag_pos.unionByName(frag_cero)
    .withColumn("suma_frag_m", F.sum("long_frag_m").over(w_jam))
    .withColumn("peso_norm", F.when(F.col("suma_frag_m") > 0, F.col("long_frag_m") / F.col("suma_frag_m"))
                              .otherwise(F.lit(1.0)))
    .withColumn("vel_kmh", F.col("speedKMH").cast("double"))
    .withColumn("nivel", F.col("level").cast("int"))
    .withColumn("retraso_jam_s", F.col("delay_minutes").cast("double") * 60)
    .withColumn("retraso_frag_s", F.col("retraso_jam_s") * F.col("peso_norm"))
    .withColumn("t_frag_s", F.when(F.col("vel_kmh") > 0, F.col("long_frag_m") / vel_ms))
    .withColumn("t_jam_s", F.when(F.col("vel_kmh") > 0, F.col("length").cast("double") / vel_ms))
    .withColumn("t_libre_s", F.col("t_jam_s") - F.col("retraso_jam_s"))
    .withColumn("vel_libre_kmh", F.when(F.col("t_libre_s") > 0,
                                        F.col("length").cast("double") / F.col("t_libre_s") * 3.6)))
frag = guardar_local(frag, "fragmentos")

# ============================================================
# 5.2 Velocidad libre (flujo libre donde no hay jam)
# ============================================================
# La mediana de cada segmento se sigue calculando (sirve para marcar el peaje y como referencia),
# pero la velocidad que se usa donde no hay jam es VEL_LIBRE_FIJA si está definida.
RUTA_VL = f"{V3}/vel_libre_segmento"
if MODO == "completo" or not existe(RUTA_VL):
    vl = (frag.filter("vel_libre_kmh IS NOT NULL").groupBy("sentido", "FROM_TO")
              .agg(F.expr("percentile_approx(vel_libre_kmh, 0.5)").alias("vel_libre_seg_kmh"),
                   F.countDistinct("clave").alias("n_trancones_vl")))
    vl_calzada = vl.groupBy("sentido").agg(F.expr("percentile_approx(vel_libre_seg_kmh, 0.5)")
                                            .alias("vel_libre_calzada_kmh"))
    (seg_sp.join(vl, ["sentido", "FROM_TO"], "left").join(vl_calzada, "sentido")
           .fillna({"n_trancones_vl": 0})
           .withColumn("vl_respaldo", F.col("vel_libre_seg_kmh").isNull() |
                                      (F.col("n_trancones_vl") < MIN_TRANCONES))
           .withColumn("vel_libre_mediana_kmh", F.when(F.col("vl_respaldo"), F.col("vel_libre_calzada_kmh"))
                                                 .otherwise(F.col("vel_libre_seg_kmh")))
           .withColumn("vel_libre_fija", F.lit(VEL_LIBRE_FIJA).cast("double"))
           .withColumn("vel_libre_ref_kmh", F.coalesce(F.col("vel_libre_fija"), F.col("vel_libre_mediana_kmh")))
           .write.mode("overwrite").parquet(RUTA_VL))
    print("Velocidad libre por segmento recalculada")
vl_seg = spark.read.parquet(RUTA_VL)
fija_guardada = vl_seg.select("vel_libre_fija").first()[0] if "vel_libre_fija" in vl_seg.columns else "—"
if fija_guardada != VEL_LIBRE_FIJA:
    raise ValueError(f"La velocidad libre guardada ({fija_guardada}) no coincide con VEL_LIBRE_FIJA "
                     f"({VEL_LIBRE_FIJA}). Corre este cuaderno con MODO = 'completo' para rehacer todo con el valor nuevo.")
vl_pd = vl_seg.toPandas()
print("Velocidad que se asigna donde no hay jam:",
      f"{VEL_LIBRE_FIJA:.0f} km/h en todo el corredor" if VEL_LIBRE_FIJA else "mediana de cada segmento")
print("Mediana histórica por calzada (solo referencia, km/h):")
print(vl_pd.groupby("sentido").vel_libre_mediana_kmh.describe()[["min", "50%", "max"]].round(1))

In [ ]:
# ============================================================
# 5.3 Estado por segmento y captura (8.3) + cierres (8.4)
# ============================================================
ref = F.broadcast(vl_seg.select("sentido", "FROM_TO", "longitud_m", "vel_libre_ref_kmh"))
claves_est = ["sentido", "FROM_TO", "ts_s", "fecha"]

est = (frag.groupBy(*claves_est)
           .agg(F.sum("long_frag_m").alias("long_cong_m"),
                F.sum("t_frag_s").alias("t_cong_s"),
                F.sum("retraso_frag_s").alias("retraso_s"),
                F.max("nivel").alias("nivel_max"),
                F.avg("vel_kmh").alias("vel_cong_media_kmh"),
                F.countDistinct("clave").alias("n_trancones"),
                F.max(F.col("es_peaje").cast("int")).alias("hay_peaje"),
                F.max(F.when(F.col("tipo_intervalo") == "reconstruido", 1).otherwise(0)).alias("hay_reconstruido"))
           .join(ref, ["sentido", "FROM_TO"])
           .withColumn("factor", F.when(F.col("long_cong_m") > F.col("longitud_m"),
                                        F.col("longitud_m") / F.col("long_cong_m")).otherwise(F.lit(1.0)))
           .withColumn("superpuesto", F.col("factor") < 1)
           .withColumn("long_cong_m", F.col("long_cong_m") * F.col("factor"))
           .withColumn("t_cong_s", F.col("t_cong_s") * F.col("factor"))
           .withColumn("cobertura", F.when(F.col("longitud_m") > 0, F.col("long_cong_m") / F.col("longitud_m"))
                                     .otherwise(F.lit(1.0)))
           .withColumn("t_libre_s", (F.col("longitud_m") - F.col("long_cong_m")) / (F.col("vel_libre_ref_kmh") / 3.6))
           .withColumn("t_total_s", F.col("t_cong_s") + F.col("t_libre_s"))
           .withColumn("vel_seg_kmh", F.when(F.col("t_total_s") > 0, F.col("longitud_m") / F.col("t_total_s") * 3.6)
                                       .otherwise(F.col("vel_cong_media_kmh")))
           .drop("longitud_m", "vel_libre_ref_kmh"))

bl_km = (fotos.filter("bloqueo")
              .select("sentido", "ts_s", "fecha", "uuid", "abs_min", "abs_max", "tipo_cierre")
              .withColumn("km", F.explode(F.sequence(km_de("abs_min"), km_de("abs_max")))))
cierres = (bl_km.join(F.broadcast(seg_km.select("sentido", "FROM_TO", "abs_ini_m", "abs_fin_m", "km")), ["sentido", "km"])
                .filter((F.col("abs_ini_m") <= F.col("abs_max")) & (F.col("abs_fin_m") >= F.col("abs_min")))
                .groupBy(*claves_est)
                .agg(F.max(F.when(F.col("tipo_cierre") == "persistente", 1).otherwise(0)).alias("cierre_persistente"),
                     F.max(F.when(F.col("tipo_cierre") == "puntual", 1).otherwise(0)).alias("cierre_puntual"),
                     F.countDistinct("uuid").alias("n_cierres")))

estado = (est.join(cierres, claves_est, "full_outer")
             .fillna({"n_trancones": 0, "n_cierres": 0, "cierre_persistente": 0, "cierre_puntual": 0})
             .withColumn("hay_trancon", F.col("n_trancones") > 0)
             .withColumn("hay_cierre", F.col("n_cierres") > 0))

RUTA_EST = f"{V3}/estado_segmento_captura"
if MODO == "completo":
    borrar(RUTA_EST)
estado.repartition("fecha").write.mode("overwrite").partitionBy("fecha").parquet(RUTA_EST)
print("Estado por segmento y captura guardado")
if VALIDAR:
    e = spark.read.parquet(RUTA_EST).filter(F.col("fecha").isin(FECHAS))
    print("5.3a Velocidad del segmento mayor que la libre (esperado: 0):",
          e.filter("hay_trancon").join(ref, ["sentido", "FROM_TO"])
           .filter("vel_seg_kmh > vel_libre_ref_kmh * 1.001").count())

## 6. Panel de 5 minutos

Una fila por **segmento activo × calzada × ventana de 5 minutos** de las fechas procesadas.

- Cada captura cae en la ventana `v = ts_s − (ts_s mod 300)`. Como capturas y jams usan el mismo reloj, **ningún jam se pierde**.
- Ventana sin captura: se completa con la captura más cercana si el hueco es de 10 minutos o menos; si no, queda *sin dato*.
- Velocidad de la ventana = promedio de sus capturas; en las capturas sin trancón, el segmento va a su velocidad libre.

Se reescriben solo las fechas procesadas (`partitionBy("fecha")`).

In [ ]:
# ============================================================
# 6. Panel de 5 minutos
# ============================================================
f_ini, f_fin = pd.Timestamp(FECHAS[0]), pd.Timestamp(FECHAS[-1])
margen = [str((f_ini - pd.Timedelta(days=1)).date()), str((f_fin + pd.Timedelta(days=1)).date())]

# ---- Capturas: las de las fechas y un día de margen a cada lado (para completar huecos) ----
cap_all = spark.read.parquet(RUTA_CAP)
lim = cap_all.agg(F.min("ts_s").alias("a"), F.max("ts_s").alias("b")).first()
v_primera, v_ultima = (lim.a // VENTANA_S) * VENTANA_S, (lim.b // VENTANA_S) * VENTANA_S
arr = np.sort(cap_all.filter(F.col("fecha").isin(FECHAS + margen)).select("ts_s").toPandas().ts_s.to_numpy())

# ---- Ventanas de las fechas procesadas (sin pasar de la primera y la última captura) ----
d0 = int(f_ini.value // 10**9)
d1 = int(f_fin.value // 10**9) + 86400 - VENTANA_S
todas = np.arange(max(d0, v_primera), min(d1, v_ultima) + VENTANA_S, VENTANA_S)
fechas_txt = pd.to_datetime(todas, unit="s").strftime("%Y-%m-%d")
todas = todas[np.isin(fechas_txt, FECHAS)]

# Capturas propias de cada ventana
v_cap = (arr // VENTANA_S) * VENTANA_S
en_rango = np.isin(v_cap, todas)
propias = pd.DataFrame({"v": v_cap[en_rango], "ts_s": arr[en_rango], "completada": False})

# Ventanas vacías: captura más cercana si el hueco es ≤ 10 min
vac = pd.DataFrame({"v": todas[~np.isin(todas, propias.v.unique())]})
vac["centro"] = vac.v + VENTANA_S // 2
idx = np.searchsorted(arr, vac.centro.to_numpy())
ok = (idx > 0) & (idx < len(arr))
ts_ant = arr[np.clip(idx - 1, 0, len(arr) - 1)]
ts_sig = arr[np.clip(idx, 0, len(arr) - 1)]
ok &= (ts_sig - ts_ant) <= MAX_HUECO_S
comp = vac[ok].copy()
comp["ts_s"] = np.where(comp.centro - ts_ant[ok] <= ts_sig[ok] - comp.centro, ts_ant[ok], ts_sig[ok])
completadas = comp[["v", "ts_s"]].assign(completada=True)
ventana_captura = pd.concat([propias, completadas], ignore_index=True)

info_v = pd.DataFrame({"v": todas}).merge(
    ventana_captura.groupby("v").agg(n_capturas=("ts_s", "size"), completada=("completada", "max")).reset_index(),
    on="v", how="left")
info_v["n_capturas"] = info_v.n_capturas.fillna(0).astype(int)
info_v["completada"] = info_v.completada.fillna(False).astype(bool)
info_v["estado_dato"] = np.select([info_v.n_capturas == 0, info_v.completada], ["sin dato", "completada"], "con captura")
print("Ventanas de 5 min en esta corrida:", f"{len(info_v):,}")
print(info_v.estado_dato.value_counts().to_string())

vc_sp     = spark.createDataFrame(ventana_captura[["v", "ts_s"]].astype("int64"))
info_v_sp = spark.createDataFrame(info_v[["v", "n_capturas", "completada", "estado_dato"]])

# ---- Estado de las capturas usadas (incluye el margen, por las ventanas completadas) ----
est_v = spark.read.parquet(RUTA_EST).filter(F.col("fecha").isin(FECHAS + margen)).drop("fecha")
agg = (est_v.join(F.broadcast(vc_sp), "ts_s")
            .groupBy("sentido", "FROM_TO", "v")
            .agg(F.sum(F.when(F.col("hay_trancon"), F.col("cobertura")).otherwise(0)).alias("sum_cob"),
                 F.sum(F.when(F.col("hay_trancon"), F.col("vel_seg_kmh"))).alias("sum_vel"),
                 F.sum(F.col("hay_trancon").cast("int")).alias("n_cap_trancon"),
                 F.sum(F.when(F.col("hay_trancon"), F.col("retraso_s")).otherwise(0)).alias("sum_ret"),
                 F.max("nivel_max").alias("nivel_max"),
                 F.max("n_trancones").alias("n_trancones"),
                 F.max("cierre_persistente").alias("cierre_persistente"),
                 F.max("cierre_puntual").alias("cierre_puntual"),
                 F.max("hay_peaje").alias("hay_peaje"),
                 F.max("hay_reconstruido").alias("hay_reconstruido")))

grilla = (vl_seg.filter("activo")
                .select("sentido", "FROM_TO", "orden_seg", "longitud_m", "abs_ini_m", "vel_libre_ref_kmh", "vl_respaldo")
                .crossJoin(F.broadcast(info_v_sp)))

n = F.col("n_capturas")
sin_dato = n == 0
n_tr = F.coalesce(F.col("n_cap_trancon"), F.lit(0))
def o_cero(c, cero=0):
    return F.when(sin_dato, None).otherwise(F.coalesce(F.col(c), F.lit(cero)))

panel = (grilla.join(agg, ["sentido", "FROM_TO", "v"], "left")
    .withColumn("cobertura", F.when(sin_dato, None).otherwise(F.coalesce(F.col("sum_cob"), F.lit(0.0)) / n))
    .withColumn("vel_kmh",   F.when(sin_dato, None).otherwise(
                                 (F.coalesce(F.col("sum_vel"), F.lit(0.0)) + (n - n_tr) * F.col("vel_libre_ref_kmh")) / n))
    .withColumn("retraso_s", F.when(sin_dato, None).otherwise(F.coalesce(F.col("sum_ret"), F.lit(0.0)) / n))
    .withColumn("nivel_max", o_cero("nivel_max"))
    .withColumn("n_trancones", o_cero("n_trancones"))
    .withColumn("cierre_persistente", o_cero("cierre_persistente"))
    .withColumn("cierre_puntual", o_cero("cierre_puntual"))
    .withColumn("hay_peaje", o_cero("hay_peaje"))
    .withColumn("hay_reconstruido", o_cero("hay_reconstruido"))
    .withColumn("estado", F.when(sin_dato, "sin dato").when(n_tr > 0, "trancon").otherwise("fluido"))
    .withColumn("ventana", F.timestamp_seconds("v"))
    .withColumn("fecha", fecha_de("v"))
    .drop("sum_cob", "sum_vel", "sum_ret", "n_cap_trancon"))

RUTA_PANEL = f"{V3}/panel_5min"
if MODO == "completo":
    borrar(RUTA_PANEL)
t0 = time.time()
panel.repartition("fecha").write.mode("overwrite").partitionBy("fecha").parquet(RUTA_PANEL)
print(f"Panel guardado ({time.time() - t0:.0f} s)")

## 7. Control y revisión rápida

Se guardan los archivos leídos y las fechas pendientes para el cuaderno 02. La revisión muestra, por fecha, cuántas ventanas tienen algún trancón y hasta qué hora llegan los datos: **ya no deberían aparecer ventanas sueltas con trancón y el resto en cero**.

In [ ]:
# ============================================================
# 7. Control de la corrida y revisión
# ============================================================
# Archivos procesados
spark.createDataFrame(archivos).write.mode("overwrite").parquet(RUTA_CTRL_ARCH)

# Fechas pendientes para el cuaderno 02 (se suman a las que no se hayan procesado todavía)
RUTA_PEND = f"{V3}/control/pendientes_tabla"
pend = set(FECHAS)
if MODO == "actualizar" and existe(RUTA_PEND):
    pend |= {r.fecha for r in spark.read.schema("fecha string").parquet(RUTA_PEND).collect()}
spark.createDataFrame(pd.DataFrame({"fecha": sorted(pend)})).write.mode("overwrite").parquet(RUTA_PEND)
print(f"Fechas pendientes para el cuaderno 02: {len(pend)} ({min(pend)} a {max(pend)})")

# Revisión de las últimas fechas procesadas
ult = FECHAS[-3:]
rev = (spark.read.parquet(RUTA_PANEL).filter(F.col("fecha").isin(ult))
            .groupBy("fecha", "v")
            .agg(F.sum((F.col("estado") == "trancon").cast("int")).alias("seg_trancon"),
                 F.first("estado_dato").alias("dato"))
            .toPandas())
rev["hora"] = pd.to_datetime(rev.v, unit="s").dt.strftime("%H:%M")
print("\nRevisión por fecha:")
for f, g in rev.groupby("fecha"):
    con = g[g.dato != "sin dato"]
    print(f"  {f}: datos de {con.hora.min()} a {con.hora.max()} · ventanas sin dato: {(g.dato == 'sin dato').sum()} · "
          f"ventanas con algún trancón: {(g.seg_trancon > 0).sum()} de {len(con)}")

g = rev[rev.fecha == ult[-1]].sort_values("v")
g = g[pd.to_datetime(g.v, unit="s").dt.hour.between(5, 21)]
print(f"\nSegmentos con trancón por hora, {ult[-1]} (debe verse una curva continua, no picos sueltos):")
print(g.assign(h=g.hora.str[:2]).groupby("h").seg_trancon.mean().round(0).astype(int).to_string())

for nombre in os.listdir(TMP_LOCAL):                 # borra los temporales (no la carpeta de Spark)
    if nombre != "spark":
        shutil.rmtree(f"{TMP_LOCAL}/{nombre}", ignore_errors=True)
print("\nListo. Sigue el cuaderno 02_TABLA_MODELADO.")

### 7b. Velocidad del corredor por hora, ciudad y vía

Misma fórmula del reloj del tablero: **suma de longitudes ÷ suma de tiempos** (tiempo = longitud ÷ velocidad del segmento), solo con los segmentos del grupo.

In [ ]:
# ============================================================
# 7b. Velocidad del corredor por hora, ciudad y vía (último día)
# ============================================================
DIA = fechas_de(RUTA_PANEL)[-1]                  # o una fecha: "2026-10-23"
pv = (spark.read.parquet(RUTA_PANEL)
           .filter((F.col("fecha") == DIA) & (F.col("estado") != "sin dato") & (F.col("vel_kmh") > 0))
           .join(F.broadcast(spark.read.parquet(RUTA_ATR).select("sentido", "FROM_TO", "city", "NomVia2")),
                 ["sentido", "FROM_TO"])
           .withColumn("hora", F.hour(F.timestamp_seconds("v")))
           .withColumn("t_h", F.col("longitud_m") / 1000 / F.col("vel_kmh"))
           .groupBy("sentido", "city", "NomVia2", "hora", "v")
           .agg((F.sum("longitud_m") / 1000).alias("km"), F.sum("t_h").alias("t_h"))
           .groupBy("sentido", "city", "NomVia2", "hora")
           .agg((F.sum("km") / F.sum("t_h")).alias("vel_kmh"))     # promedio de la hora, ponderado por tiempo
           .toPandas())
pv["calzada"] = pv.sentido.map(NOMBRES)
print(f"Velocidad (km/h) por ciudad y vía · {DIA} · flujo libre = {VEL_LIBRE_FIJA} km/h")
display(pv.pivot_table(index=["calzada", "city", "NomVia2"], columns="hora", values="vel_kmh").round(1))